# 🦆 Entraînement du modèle de détection de marque nasale (YOLO26)

Ce notebook fait tourner l'entraînement sur un **GPU gratuit** fourni par Google Colab, au lieu de ton PC local — plus rapide et ça ne fait pas chauffer ta machine.

**Avant de commencer :**
1. En haut du notebook : **Exécution** → **Modifier le type d'exécution** → sélectionne **GPU** (T4 gratuit) → Enregistrer
2. Sur ton PC : va dans ton dossier `canard-detection/`, sélectionne le dossier `dataset/` (celui qui contient `data.yaml`, `images/`, `labels/`) et compresse-le en `.zip`
3. Exécute les cellules ci-dessous **dans l'ordre**, une par une (bouton ▶️ à gauche de chaque cellule, ou Maj+Entrée)

## 1. Installer Ultralytics

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

La dernière ligne doit afficher `CUDA` avec un GPU détecté (ex. Tesla T4). Si tu vois `CPU`, retourne dans **Exécution → Modifier le type d'exécution** et vérifie que GPU est bien sélectionné, puis relance cette cellule.

## 2. Importer ton dataset

Cette cellule ouvre un bouton pour uploader ton fichier `dataset.zip` (celui préparé à l'étape 2 ci-dessus).

In [ ]:
from google.colab import files
import zipfile
import os

print("Sélectionne ton fichier dataset.zip...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]

os.makedirs("/content/project", exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/project/dataset")

print("\n✅ Dataset extrait dans /content/project/dataset")
!find /content/project/dataset -maxdepth 2

**Vérifie la sortie ci-dessus** : tu dois voir `data.yaml`, `images/train`, `images/val`, `labels/train`, `labels/val` directement dans `/content/project/dataset`.

⚠️ Si tu vois un dossier en double (ex. `/content/project/dataset/dataset/...`), c'est que ton zip contenait déjà un dossier `dataset` à l'intérieur — dans ce cas, exécute la cellule suivante pour corriger automatiquement, sinon passe directement à l'étape 3.

In [ ]:
# À exécuter UNIQUEMENT si tu as un dossier en double (dataset/dataset/...)
import shutil

nested = "/content/project/dataset/dataset"
if os.path.exists(nested):
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), "/content/project/dataset")
    shutil.rmtree(nested)
    print("✅ Dossier en double corrigé.")
else:
    print("Pas de dossier en double détecté, rien à faire.")

## 3. Corriger le chemin dans data.yaml

Comme sur ton PC, on force un chemin absolu pour éviter tout souci de résolution de chemin. On en profite aussi pour **vérifier que le dataset est cohérent** avant de lancer plusieurs heures d'entraînement pour rien (classes déclarées, présence des images/labels, correspondance des paires).

In [ ]:
import yaml
from pathlib import Path

data_yaml_path = "/content/project/dataset/data.yaml"

with open(data_yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = "/content/project/dataset"

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print("data.yaml mis à jour :")
print(open(data_yaml_path).read())

# --- Vérifications de cohérence du dataset ---
print("=" * 60)
print("🔍 VÉRIFICATION DU DATASET")
print("=" * 60)

nc = cfg.get("nc")
names = cfg.get("names")
print(f"Nombre de classes (nc) : {nc}")
print(f"Classes (names)        : {names}")

if nc is None or names is None:
    print("⚠️  'nc' ou 'names' absent de data.yaml — vérifie ce fichier avant de continuer.")
elif isinstance(names, (list, dict)) and len(names) != nc:
    print(f"⚠️  Incohérence : nc={nc} mais {len(names)} classes listées dans 'names'.")
else:
    print("✓ nc et names cohérents")

dataset_root = Path(cfg["path"])
anomalies = 0

for split in ("train", "val"):
    split_key = cfg.get(split, f"images/{split}")
    images_dir = dataset_root / split_key if not str(split_key).startswith("/") else Path(split_key)
    labels_dir = dataset_root / "labels" / split

    if not images_dir.exists():
        print(f"❌ Dossier images introuvable pour '{split}' : {images_dir}")
        anomalies += 1
        continue

    images = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    labels = sorted(labels_dir.glob("*.txt")) if labels_dir.exists() else []

    images_sans_label = [img for img in images if not (labels_dir / f"{img.stem}.txt").exists()]

    print(f"\n{split.upper()} : {len(images)} images, {len(labels)} fichiers labels")
    if images_sans_label:
        print(f"⚠️  {len(images_sans_label)} image(s) sans fichier label correspondant (ex: {images_sans_label[0].name})")
        anomalies += 1
    if len(images) == 0:
        print(f"❌ Aucune image trouvée pour le split '{split}'")
        anomalies += 1

print("\n" + "=" * 60)
if anomalies == 0:
    print("✅ Dataset cohérent, prêt pour l'entraînement")
else:
    print(f"⚠️  {anomalies} anomalie(s) détectée(s) — corrige-les avant de lancer un entraînement de plusieurs heures")
print("=" * 60)

## 4. (Optionnel) Reprendre depuis un modèle déjà entraîné

Si tu as déjà un `best.pt` d'un entraînement précédent (fait sur ton PC ou ici) et que tu veux repartir de là plutôt que de zéro, uploade-le avec la cellule ci-dessous. **Sinon, passe directement à l'étape 5** — le notebook partira automatiquement de `yolo26n.pt` (pré-entraîné COCO).

In [ ]:
# À exécuter UNIQUEMENT si tu veux repartir d'un best.pt existant
from google.colab import files

print("Sélectionne ton fichier best.pt...")
uploaded_model = files.upload()
start_weights = list(uploaded_model.keys())[0]
print(f"\n✅ On repartira de : {start_weights}")

## 5. (Recommandé) Sauvegarder l'entraînement sur Google Drive

Une session Colab gratuite peut se déconnecter (inactivité, limite ~12h). Avec 300 epochs, l'entraînement peut durer plusieurs heures : si la session se coupe et que les résultats ne sont que dans `/content` (stockage éphémère de la VM), **tout est perdu**.

Cette cellule est optionnelle mais fortement conseillée : elle monte ton Google Drive et fait écrire les checkpoints et résultats directement dessus, pour ne rien perdre en cas de déconnexion. Si tu préfères ne pas utiliser Drive, réponds "n" — l'entraînement se fera alors uniquement dans `/content/runs` (à télécharger toi-même avant la fin de session).

In [ ]:
use_drive = input("Sauvegarder l'entraînement sur Google Drive ? (o/N) : ").strip().lower() == "o"

if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    TRAIN_PROJECT_DIR = "/content/drive/MyDrive/canard-detection/runs"
    os.makedirs(TRAIN_PROJECT_DIR, exist_ok=True)
    print(f"✅ Les résultats seront sauvegardés sur Drive : {TRAIN_PROJECT_DIR}")
else:
    TRAIN_PROJECT_DIR = "/content/runs"
    print(f"ℹ️  Les résultats resteront dans le stockage temporaire de la session : {TRAIN_PROJECT_DIR}")
    print("   Pense à bien exécuter les cellules de téléchargement (étape 9) avant la fin de la session.")

## 6. Lancer l'entraînement

Ajuste `EPOCHS`, `IMG_SIZE`, `BATCH` si besoin. Sur GPU T4, tu peux monter le batch beaucoup plus haut que sur CPU (16 ou 32). Un `SEED` fixe est utilisé pour que l'entraînement soit reproductible.

In [ ]:
from ultralytics import YOLO

EPOCHS = 300
IMG_SIZE = 960
BATCH = 16
SEED = 42

# Reprend le best.pt uploadé à l'étape 4 si présent, sinon part du modèle pré-entraîné COCO
if "start_weights" in globals():
    print(f"▶️  Reprise depuis un modèle existant : {start_weights}")
    model = YOLO(start_weights)
else:
    print("▶️  Départ depuis le modèle pré-entraîné COCO : yolo26n.pt")
    model = YOLO("yolo26n.pt")

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=30,
    seed=SEED,
    project=TRAIN_PROJECT_DIR,
    name="marque_nasale_colab",
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    close_mosaic=15,
)

RUN_DIR = results.save_dir  # dossier réel du run (gère les suffixes marque_nasale_colab2, 3, ... en cas de ré-exécution)
print(f"\n✅ Entraînement terminé. Résultats dans : {RUN_DIR}")

⚠️ **Attention aux sessions Colab gratuites** : elles se coupent automatiquement après un certain temps d'inactivité ou une durée maximale (~12h). Reste sur l'onglet de temps en temps pour éviter une déconnexion, et ne ferme pas l'onglet pendant l'entraînement. Si tu as activé la sauvegarde sur Drive à l'étape 5, les checkpoints (`last.pt`) sont régulièrement écrits sur Drive : en cas de coupure, tu peux relancer l'étape 4 avec ce `last.pt` pour reprendre l'entraînement plutôt que de tout recommencer.

## 7. Évaluer le modèle

Cette cellule calcule les métriques de validation (précision, rappel, mAP) et génère la matrice de confusion — indispensable pour juger si le modèle est réellement utilisable avant de l'intégrer au pipeline.

In [ ]:
metrics = model.val()

print("=" * 60)
print("📊 MÉTRIQUES DE VALIDATION")
print("=" * 60)
print(f"Precision (P)      : {metrics.box.mp:.3f}")
print(f"Recall (R)         : {metrics.box.mr:.3f}")
print(f"mAP50              : {metrics.box.map50:.3f}")
print(f"mAP50-95           : {metrics.box.map:.3f}")
print(f"\n📂 Matrice de confusion et courbes PR sauvegardées dans : {metrics.save_dir}")

if metrics.box.map50 < 0.5:
    print("\n⚠️  mAP50 faible (<0.5) : envisage plus d'epochs, plus de données annotées, ou revoir la qualité des labels.")

## 8. Regarder les courbes d'entraînement

In [ ]:
import glob
from IPython.display import Image, display

# Utilise le dossier du run qui vient de se terminer si disponible, sinon cherche le plus récent
results_png = os.path.join(RUN_DIR, "results.png") if "RUN_DIR" in globals() else None

if not results_png or not os.path.exists(results_png):
    chemins = glob.glob(f"{TRAIN_PROJECT_DIR}/marque_nasale_colab*/results.png") if "TRAIN_PROJECT_DIR" in globals() else glob.glob("/content/runs/marque_nasale_colab*/results.png")
    results_png = sorted(chemins)[-1] if chemins else None

if results_png:
    print(f"Affichage de : {results_png}")
    display(Image(filename=results_png))
else:
    print("Graphique introuvable. Vérifiez l'arborescence à gauche dans Colab.")

## 9. Télécharger le modèle entraîné (best.pt)

Récupère ce fichier et remplace/ajoute-le dans `runs\detect\<nom_du_run>\weights\best.pt` sur ton PC pour continuer à utiliser `03_predict.py` normalement, ou pour repartir de ce point la prochaine fois.

*(Si tu as sauvegardé sur Google Drive à l'étape 5, le fichier y est déjà disponible directement — ce téléchargement est surtout utile si tu n'as pas utilisé Drive.)*

In [ ]:
from google.colab import files

# Résout dynamiquement le chemin du dernier run (gère marque_nasale_colab, marque_nasale_colab2, ...)
best_pt_path = os.path.join(RUN_DIR, "weights", "best.pt") if "RUN_DIR" in globals() else None

if not best_pt_path or not os.path.exists(best_pt_path):
    base_dir = TRAIN_PROJECT_DIR if "TRAIN_PROJECT_DIR" in globals() else "/content/runs"
    candidats = sorted(glob.glob(f"{base_dir}/marque_nasale_colab*/weights/best.pt"))
    best_pt_path = candidats[-1] if candidats else None

if best_pt_path and os.path.exists(best_pt_path):
    print(f"Téléchargement de : {best_pt_path}")
    files.download(best_pt_path)
else:
    print("❌ best.pt introuvable. Vérifie que l'entraînement (étape 6) s'est bien terminé.")

## 10. (Optionnel) Sauvegarder aussi les résultats/courbes

In [ ]:
import shutil

run_dir_to_archive = RUN_DIR if "RUN_DIR" in globals() and os.path.exists(RUN_DIR) else None

if not run_dir_to_archive:
    base_dir = TRAIN_PROJECT_DIR if "TRAIN_PROJECT_DIR" in globals() else "/content/runs"
    candidats = sorted(glob.glob(f"{base_dir}/marque_nasale_colab*"))
    run_dir_to_archive = candidats[-1] if candidats else None

if run_dir_to_archive:
    archive_path = shutil.make_archive("/content/resultats_entrainement", "zip", run_dir_to_archive)
    files.download(archive_path)
else:
    print("❌ Dossier de run introuvable, rien à archiver.")

## 11. Tester sur un dossier de nouvelles images

Cette section est **indépendante de l'entraînement** : tu peux la lancer dans une session Colab toute neuve, sans avoir besoin de ré-entraîner quoi que ce soit. Il te faut juste :

- ton fichier `best.pt` (téléchargé à l'étape 9, ou celui déjà en mémoire si tu enchaînes dans la même session)
- un dossier avec tes nouvelles images, **compressé en `.zip`**

👉 **Si tu viens de finir l'entraînement dans cette même session**, la cellule 11.1 ci-dessous s'en rend compte automatiquement et ne te redemandera pas d'uploader `best.pt` — mais tu peux quand même le faire si tu veux tester un autre modèle.

In [ ]:
# 11.1 Charger le modèle
from ultralytics import YOLO
from google.colab import files

model_deja_charge = "model" in globals()

if model_deja_charge:
    print("ℹ️ Un modèle est déjà chargé en mémoire dans cette session.")
    reponse = input("Uploader quand même un autre best.pt ? (o/N) : ").strip().lower()
else:
    reponse = "o"

if reponse == "o":
    print("Sélectionne ton fichier best.pt...")
    uploaded_model = files.upload()
    model_path = list(uploaded_model.keys())[0]
    model = YOLO(model_path)
    print(f"\n✅ Modèle chargé depuis : {model_path}")
else:
    print("✅ On garde le modèle déjà chargé en mémoire.")

In [ ]:
# 11.2 Importer ton dossier de nouvelles images (compressé en .zip)
from google.colab import files
import zipfile
import os
import shutil

TEST_DIR = "/content/test_images"

if os.path.exists(TEST_DIR):
    shutil.rmtree(TEST_DIR)
os.makedirs(TEST_DIR, exist_ok=True)

print("Sélectionne ton dossier d'images compressé en .zip...")
uploaded_tests = files.upload()
zip_name = list(uploaded_tests.keys())[0]

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(TEST_DIR)

# Si le zip contenait un dossier englobant (ex. test_images/test_images/...), on aplatit d'un niveau
entries = os.listdir(TEST_DIR)
if len(entries) == 1 and os.path.isdir(os.path.join(TEST_DIR, entries[0])):
    nested = os.path.join(TEST_DIR, entries[0])
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), TEST_DIR)
    shutil.rmtree(nested)

images = [f for f in os.listdir(TEST_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"\n✅ {len(images)} image(s) trouvée(s) dans {TEST_DIR}")

In [ ]:
# 11.3 Lancer les prédictions sur le dossier
CONF_THRESHOLD = 0.25   # seuil de confiance, baisse-le pour voir plus de détections faibles
PREDICT_IMG_SIZE = 960  # doit correspondre à l'IMG_SIZE utilisé à l'entraînement (étape 6)

results = model.predict(
    source="/content/test_images",
    conf=CONF_THRESHOLD,
    imgsz=PREDICT_IMG_SIZE,
    save=True,
    project="/content/runs",
    name="test_predictions",
    exist_ok=True,
)

for r in results:
    print(f"\n{os.path.basename(r.path)}")
    if r.boxes is None or len(r.boxes) == 0:
        print("  Aucune détection.")
        continue
    for box in r.boxes:
        conf = float(box.conf[0])
        print(f"  marque_nasale — confiance {conf:.2f}")

print(f"\n📂 Images annotées enregistrées dans : /content/runs/test_predictions")

In [ ]:
# 11.4 Visualiser un aperçu des résultats
from IPython.display import Image, display
import glob

for img_path in sorted(glob.glob("/content/runs/test_predictions/*.jpg")):
    print(os.path.basename(img_path))
    display(Image(filename=img_path, width=500))

In [ ]:
# 11.5 Télécharger le dossier complet des images annotées
import shutil

shutil.make_archive("/content/predictions_marque_nasale", "zip", "/content/runs/test_predictions")
files.download("/content/predictions_marque_nasale.zip")

print("✅ Dézippe le fichier téléchargé : tu retrouveras un dossier avec toutes tes images, boîtes incluses.")